In [1]:
import numpy as np
from scipy.spatial import cKDTree
import trimesh


def nearest_vertices_in_fine_mesh(V_coarse, V_fine):
    tree = cKDTree(V_fine)
    distances, indices = tree.query(V_coarse)
    return indices, distances


def bounding_box_center_and_size(V):
    """
    Returns center and longest side length of axis-aligned bounding box.
    """
    vmin = V.min(axis=0)
    vmax = V.max(axis=0)

    center = 0.5 * (vmin + vmax)
    size = np.max(vmax - vmin)

    return center, size


def normalize_to_match_bbox(V_src, src_center, src_size,
                            target_center, target_size):
    """
    Uniformly scale + translate V_src so its bbox matches target.
    """

    scale = target_size / src_size

    Vn = (V_src - src_center) * scale + target_center

    return Vn, scale


def remesh_vertices_from_fine(
    coarse_obj_path,
    fine_obj_path,
    fine_deformed_obj_path,
    output_obj_path,
):

    # --- Load meshes ---
    coarse = trimesh.load(coarse_obj_path, process=False)
    fine = trimesh.load(fine_obj_path, process=False)
    fine_deformed = trimesh.load(fine_deformed_obj_path, process=False)

    if not isinstance(coarse, trimesh.Trimesh):
        raise ValueError("Coarse OBJ did not load as a single mesh.")
    if not isinstance(fine, trimesh.Trimesh):
        raise ValueError("Fine OBJ did not load as a single mesh.")
    if not isinstance(fine_deformed, trimesh.Trimesh):
        raise ValueError("Fine-deformed OBJ did not load as a single mesh.")

    V_coarse = np.asarray(coarse.vertices)
    V_fine = np.asarray(fine.vertices)
    V_fine_def = np.asarray(fine_deformed.vertices)

    # --------------------------------------------------
    # Normalize fine + fine_deformed to coarse bbox
    # --------------------------------------------------

    coarse_center, coarse_size = bounding_box_center_and_size(V_coarse)
    fine_center, fine_size = bounding_box_center_and_size(V_fine)

    V_fine_norm, scale = normalize_to_match_bbox(
        V_fine,
        fine_center,
        fine_size,
        coarse_center,
        coarse_size,
    )

    # apply EXACT same transform to deformed fine
    V_fine_def_norm = (V_fine_def - fine_center) * scale + coarse_center

    # --------------------------------------------------
    # Nearest neighbor in normalized space
    # --------------------------------------------------

    idx, dist = nearest_vertices_in_fine_mesh(
        V_coarse,
        V_fine_norm,
    )

    # --------------------------------------------------
    # Build new mesh from normalized fine_deformed
    # --------------------------------------------------

    V_new = V_fine_def_norm[idx]

    new_mesh = trimesh.Trimesh(
        vertices=V_new,
        faces=coarse.faces,
        process=False,
    )

    new_mesh.export(output_obj_path)

    print(f"Saved remeshed OBJ to: {output_obj_path}")
    print(f"Mean snapping distance: {dist.mean():.6f}")
    print(f"Max snapping distance:  {dist.max():.6f}")
    print(f"Applied uniform scale factor: {scale:.6f}")


# -----------------------------
# Example usage
# -----------------------------

if __name__ == "__main__":

    coarse_path = "data/surfaces/lion-reference300.obj"
    fine_path = "data/surfaces/corresponding-shapes/lion-reference.obj"
    fine_deformed_path = "data/surfaces/corresponding-shapes/lion-stretch.obj"
    output_path = "data/surfaces/lion-stretch300.obj"

    remesh_vertices_from_fine(
        coarse_path,
        fine_path,
        fine_deformed_path,
        output_path,
    )


Saved remeshed OBJ to: data/surfaces/lion-stretch300.obj
Mean snapping distance: 0.139261
Max snapping distance:  0.409212
Applied uniform scale factor: 1.278723
